### Import Dependencies


In [188]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import mlflow
from pathlib import Path
import gc  # garbage collector

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import TargetEncoder

# ml models
from sklearn.ensemble import HistGradientBoostingClassifier

# ensembles
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool

# dl models
from pytabkit import (
    RealMLP_TD_Classifier, # noqa
    Resnet_RTDL_D_Classifier # noqa
)

# import metrics
from sklearn.metrics import log_loss, roc_auc_score, recall_score, precision_score

pd.set_option("display.max_columns", None)

### Read Data


In [189]:
BASE_DIR = Path("../../data/raw")

app_train = pd.read_csv(BASE_DIR / "applications_train.csv")
app_test = pd.read_csv(BASE_DIR / "applications_test.csv")

In [190]:
print(f"Shape of applications_train: {app_train.shape}")
print(f"Shape of applications_test: {app_test.shape}")

Shape of applications_train: (292135, 122)
Shape of applications_test: (15376, 122)


In [191]:
# lower casing the feature names
app_train.columns = app_train.columns.str.lower()
app_test.columns = app_test.columns.str.lower()

X = app_train.drop("target", axis=1)
y = app_train["target"]

X_test = app_test.drop("target", axis=1)
y_test = app_test["target"]

### Configs


In [192]:
class Config:
    seed = 42
    n_folds = 5
    n_classes = 2

In [193]:
class PARAMS:
    # Histgbm
    hgb = {
        "loss": "log_loss",
        "learning_rate": 0.03,
        "max_iter": 800,
        "max_depth": 5,
        "max_features": 0.8,
        "early_stopping": False,
        "validation_fraction": None,
        "verbose": 0,
        "random_state": Config.seed,
        "categorical_features": "from_dtype",
        "class_weight": "balanced", # giving same cost to each class
    }

    lgbm = {
        "objective": "binary",
        "boosting_type": "gbdt",
        "metric": "auc",
        "learning_rate": 0.03,
        "num_leaves": 31,
        "max_depth": -1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": Config.seed,
        "n_jobs": -1,
        "importance_type": "gain",
        "verbose": -1,
        "is_unbalance": True,
    }

    xgb = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "learning_rate": 0.03,
        "max_depth": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": Config.seed,
        "n_jobs": -1,
        "device": "cuda",
    }

    cb = {
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "iterations": 1000,
        "learning_rate": 0.03,
        "depth": 5,
        "l2_leaf_reg": 3.0,
        "random_seed": Config.seed,
        "early_stopping_rounds": 100,
        "thread_count": -1,
        "verbose": False,
        "bootstrap_type": "Bayesian",
        "use_best_model": True,
        "task_type": "GPU",
        "auto_class_weights": "Balanced"
    }

### Data Cleaning


In [194]:
def replace_XNA_XAP(table):

    # Replace all values of 'XNA', 'XAP' with np.nan
    return table.replace(to_replace=["XNA", "XAP"], value=np.nan)

In [195]:
# Several features have entries with values that can best
# be interpreted as np.nan:

# Replace all entries of 365243 in 'DAYS_EMPLOYED' with nan
X["days_employed"] = X["days_employed"].replace(365243, np.nan)
X_test["days_employed"] = X_test["days_employed"].replace(365243, np.nan)

# Replace all entries of 0 in 'DAYS_LAST_PHONE_CHANGE' with nan
X["days_last_phone_change"] = X["days_last_phone_change"].replace(0, np.nan)
X_test["days_last_phone_change"] = X_test["days_last_phone_change"].replace(0, np.nan)

# Replace all entries of 'XNA' or 'XAP' in main data table with np.nan
# (Such entries should be confined to the features 'CODE_GENDER'
# and 'ORGANIZATION_TYPE'.)
X = replace_XNA_XAP(X)
X_test = replace_XNA_XAP(X_test)

# Two rows in training table have a value of 'Unknown' for
# 'name_family_status', but no rows in test table do.
X["name_family_status"] = X["name_family_status"].replace("Unknown", np.nan)
X_test["name_family_status"] = X_test["name_family_status"].replace("Unknown", np.nan)

# Five rows in training table have a value of 'Maternity leave' for
# 'name_income_type', but no rows in test table do.
X["name_income_type"] = X["name_income_type"].replace("Maternity leave", np.nan)
X_test["name_income_type"] = X_test["name_income_type"].replace(
    "Maternity leave", np.nan
)

# No rows in training table have -1 for 'region_rating_client_w_city'
# but at least one row in test table does.
X["region_rating_client_w_city"] = X["region_rating_client_w_city"].replace(-1, np.nan)
X_test["region_rating_client_w_city"] = X_test["region_rating_client_w_city"].replace(
    -1, np.nan
)

In [196]:
# keeping nulls as it is coz tree models handles them naturally
X.isna().sum()[X.isna().sum() > 10].to_frame().T

,amt_goods_price,name_type_suite,days_employed,own_car_age,occupation_type,organization_type,ext_source_1,ext_source_2,ext_source_3,apartments_avg,basementarea_avg,years_beginexpluatation_avg,years_build_avg,commonarea_avg,elevators_avg,entrances_avg,floorsmax_avg,floorsmin_avg,landarea_avg,livingapartments_avg,livingarea_avg,nonlivingapartments_avg,nonlivingarea_avg,apartments_mode,basementarea_mode,years_beginexpluatation_mode,years_build_mode,commonarea_mode,elevators_mode,entrances_mode,floorsmax_mode,floorsmin_mode,landarea_mode,livingapartments_mode,livingarea_mode,nonlivingapartments_mode,nonlivingarea_mode,apartments_medi,basementarea_medi,years_beginexpluatation_medi,years_build_medi,commonarea_medi,elevators_medi,entrances_medi,floorsmax_medi,floorsmin_medi,landarea_medi,livingapartments_medi,livingarea_medi,nonlivingapartments_medi,nonlivingarea_medi,fondkapremont_mode,housetype_mode,totalarea_mode,wallsmaterial_mode,emergencystate_mode,obs_30_cnt_social_circle,def_30_cnt_social_circle,obs_60_cnt_social_circle,def_60_cnt_social_circle,days_last_phone_change,amt_req_credit_bureau_hour,amt_req_credit_bureau_day,amt_req_credit_bureau_week,amt_req_credit_bureau_mon,amt_req_credit_bureau_qrt,amt_req_credit_bureau_year
0,265,1212,52585,192773,91502,52585,164734,621,57919,148223,170862,142485,194258,204078,155625,147052,145330,198201,173422,199681,146611,202806,161132,148223,170862,142485,194258,204078,155625,147052,145330,198201,173422,199681,146611,202806,161132,148223,170862,142485,194258,204078,155625,147052,145330,198201,173422,199681,146611,202806,161132,199774,146527,140989,148480,138451,975,975,975,975,35792,39434,39434,39434,39434,39434,39434


### Data Preprocessing


In [197]:
binary_cols = [
    "code_gender",
    "flag_own_car",
    "flag_own_realty",
    "emergencystate_mode",
]

ohe_cols = [
    "name_contract_type",
    "name_type_suite",
    "name_income_type",
    "name_family_status",
    "name_housing_type",
    "occupation_type",
    "weekday_appr_process_start",
    "organization_type",
    "fondkapremont_mode",
    "housetype_mode",
    "wallsmaterial_mode",
]

ordinal_cols = ["name_education_type"]

In [198]:
# mappings
code_gender_map = {"M": 0, "F": 1}

flag_own_car_map = {"Y": 1, "N": 0}

flag_own_realty_map = {"Y": 1, "N": 0}

emergencystate_mode_map = {"Yes": 1, "No": 0}

name_education_type_mapping = {
    "Lower secondary": 0,
    "Secondary / secondary special": 1,
    "Incomplete higher": 2,
    "Higher education": 3,
    "Academic degree": 4,
}

In [199]:
# feature encoding
X = (
    X.assign(
        code_gender=lambda df: df["code_gender"].map(code_gender_map),
        flag_own_car=lambda df: df["flag_own_car"].map(flag_own_car_map),
        flag_own_realty=lambda df: df["flag_own_realty"].map(flag_own_realty_map),
        emergencystate_mode=lambda df: df["emergencystate_mode"].map(
            emergencystate_mode_map
        ),
        name_education_type=lambda df: df["name_education_type"].map(
            name_education_type_mapping
        ),
    )
)

X_test = (
    X_test.assign(
        code_gender=lambda df: df["code_gender"].map(code_gender_map),
        flag_own_car=lambda df: df["flag_own_car"].map(flag_own_car_map),
        flag_own_realty=lambda df: df["flag_own_realty"].map(flag_own_realty_map),
        emergencystate_mode=lambda df: df["emergencystate_mode"].map(
            emergencystate_mode_map
        ),
        name_education_type=lambda df: df["name_education_type"].map(
            name_education_type_mapping
        ),
    )
)

#### Feature Engineering

In [200]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:

    # 'HAS_CHILDREN': a binary feature that indicates whether or
    # not a borrower has one or more children.
    df["has_children"] = df["cnt_children"].map(lambda x: 1 if x > 0 else 0)

    # 'HAS_JOB': a binary feature that indicates whether or not
    # a borrower was employed when their application was submitted.
    df["has_job"] = df["days_employed"].map(lambda x: 1 if x < 0 else 0)

    # Sums
    df["sum_amt_income_total_amt_annuity"] = (
        df["amt_income_total"] + df["amt_annuity"]
    )

    df["total_enquiries_credit_bureau"] = df[
        [
            "amt_req_credit_bureau_day",
            "amt_req_credit_bureau_hour",
            "amt_req_credit_bureau_week",
            "amt_req_credit_bureau_mon",
            "amt_req_credit_bureau_qrt",
            "amt_req_credit_bureau_year",
        ]
    ].sum(axis=1)

    # Differences
    df["diff_amt_credit_amt_goods_price"] = (
        df["amt_credit"] - df["amt_goods_price"]
    )

    df["diff_amt_annuity_amt_goods_price"] = (
        df["amt_annuity"] - df["amt_goods_price"]
    )

    df["diff_amt_income_total_amt_annuity"] = (
        df["amt_income_total"] - df["amt_annuity"]
    )

    df["cnt_adult_fam_member"] = (
        df["cnt_fam_members"] - df["cnt_children"]
    )

    df["diff_obs_30_cnt_social_circle_obs_60_cnt_social_circle"] = (
        df["obs_30_cnt_social_circle"] - df["obs_60_cnt_social_circle"]
    )

    df["diff_def_30_cnt_social_circle_def_60_cnt_social_circle"] = (
        df["def_30_cnt_social_circle"] - df["def_60_cnt_social_circle"]
    )

    # ratio
    df["ratio_amt_credit_to_amt_annuity"] = (
        df["amt_credit"] / df["amt_annuity"].replace(0, np.nan)
    )

    df["ratio_amt_credit_to_cnt_adult_fam_member"] = (
        df["amt_credit"] / df["cnt_adult_fam_member"].replace(0, np.nan)
    )

    df["ratio_amt_income_total_to_amt_annuity"] = (
        df["amt_income_total"] / df["amt_annuity"].replace(0, np.nan)
    )

    df["amt_income_total_per_adult_fam_member"] = (
        df["amt_income_total"]
        / df["cnt_adult_fam_member"].replace(0, np.nan)
    )

    df["ratio_amt_goods_price_to_livingarea_avg"] = (
        df["amt_goods_price"] / df["livingarea_avg"].replace(0, np.nan)
    )

    df["ratio_amt_goods_price_to_landarea_avg"] = (
        df["amt_goods_price"] / df["landarea_avg"].replace(0, np.nan)
    )

    df["ratio_amt_goods_price_to_floorsmax_avg_avg"] = (
        df["amt_goods_price"] / df["floorsmax_avg"].replace(0, np.nan)
    )

    df["ratio_amt_goods_price_to_livingapartments_avg"] = (
        df["amt_goods_price"]
        / df["livingapartments_avg"].replace(0, np.nan)
    )

    df["ratio_amt_goods_price_to_years_build_avg"] = (
        df["amt_goods_price"] / df["years_build_avg"].replace(0, np.nan)
    )

    df["ratio_amt_goods_price_to_days_employed"] = (
        df["amt_goods_price"] / df["days_employed"].replace(0, np.nan)
    )

    df["ratio_amt_goods_price_to_cnt_children"] = (
        df["amt_goods_price"] / df["cnt_children"].replace(0, np.nan)
    )

    df["ratio_amt_goods_price_to_sum_amt_income_total_amt_annuity"] = (
        df["amt_goods_price"]
        / df["sum_amt_income_total_amt_annuity"].replace(0, np.nan)
    )

    df["ratio_amt_annuity_to_livingarea_avg"] = (
        df["amt_annuity"] / df["livingarea_avg"].replace(0, np.nan)
    )

    df["ratio_amt_annuity_to_days_employed"] = (
        df["amt_annuity"] / df["days_employed"].replace(0, np.nan)
    )

    df["ratio_amt_annuity_to_cnt_children"] = (
        df["amt_annuity"] / df["cnt_children"].replace(0, np.nan)
    )

    df["ratio_amt_annuity_to_cnt_adult_fam_member"] = (
        df["amt_annuity"] / df["cnt_adult_fam_member"].replace(0, np.nan)
    )

    df["ratio_ext_source_3_to_region_population_relative"] = (
        df["ext_source_3"]
        / df["region_population_relative"].replace(0, np.nan)
    )

    df["ratio_days_last_phone_change_to_days_registration"] = (
        df["days_last_phone_change"]
        / df["days_registration"].replace(0, np.nan)
    )

    df["pctg_fam_children"] = (
        df["cnt_children"] / df["cnt_fam_members"].replace(0, np.nan)
    )

    df["pctg_enquiries_hour"] = (
        df["amt_req_credit_bureau_hour"]
        / df["total_enquiries_credit_bureau"].replace(0, np.nan)
    )

    df["pctg_enquiries_day"] = (
        df["amt_req_credit_bureau_day"]
        / df["total_enquiries_credit_bureau"].replace(0, np.nan)
    )

    df["pctg_enquiries_week"] = (
        df["amt_req_credit_bureau_week"]
        / df["total_enquiries_credit_bureau"].replace(0, np.nan)
    )

    df["pctg_enquiries_mon"] = (
        df["amt_req_credit_bureau_mon"]
        / df["total_enquiries_credit_bureau"].replace(0, np.nan)
    )

    df["pctg_enquiries_qrt"] = (
        df["amt_req_credit_bureau_qrt"]
        / df["total_enquiries_credit_bureau"].replace(0, np.nan)
    )

    df["pctg_enquiries_year"] = (
        df["amt_req_credit_bureau_year"]
        / df["total_enquiries_credit_bureau"].replace(0, np.nan)
    )

    # Based on EXT_SOURCES features
    df["ext_sources_weighted_sum"] = (
        df["ext_source_3"] * 5
        + df["ext_source_1"] * 3
        + df["ext_source_2"]
    )

    df["ext_sources_weighted_avg"] = (
        df["ext_source_3"] * 5
        + df["ext_source_1"] * 3
        + df["ext_source_2"]
    ) / 3

    # Ratios
    df["ratio_amt_credit_to_amt_goods_price"] = (
        df["amt_credit"]
        / df["amt_goods_price"].replace(0, np.nan)
    )

    df["ratio_amt_credit_to_amt_income_total"] = (
        df["amt_credit"]
        / df["amt_income_total"].replace(0, np.nan)
    )

    df["ratio_amt_credit_to_cnt_fam_members"] = (
        df["amt_credit"]
        / df["cnt_fam_members"].replace(0, np.nan)
    )

    df["ratio_amt_credit_to_cnt_children"] = (
        df["amt_credit"]
        / (1 + df["cnt_children"])
    )

    df["ratio_amt_income_total_to_amt_credit"] = (
        df["amt_income_total"]
        / df["amt_credit"].replace(0, np.nan)
    )

    df["ratio_amt_income_total_to_cnt_children"] = (
        df["amt_income_total"]
        / (1 + df["cnt_children"])
    )

    df["ratio_amt_annuity_to_amt_income_total"] = (
        df["amt_annuity"]
        / df["amt_income_total"].replace(0, np.nan)
    )

    df["ratio_amt_annuity_amt_credit"] = (
        df["amt_annuity"]
        / df["amt_credit"].replace(0, np.nan)
    )

    df["ratio_children_to_adults"] = (
        df["cnt_children"]
        / df["cnt_adult_fam_member"].replace(0, np.nan)
    )

    df["ratio_own_car_age_to_days_birth"] = (
        df["own_car_age"]
        / df["days_birth"].replace(0, np.nan)
    )

    df["ratio_own_car_age_to_days_employed"] = (
        df["own_car_age"]
        / df["days_employed"].replace(0, np.nan)
    )

    df["ratio_days_last_phone_change_to_days_birth"] = (
        df["days_last_phone_change"]
        / df["days_birth"].replace(0, np.nan)
    )

    df["ratio_days_last_phone_change_to_days_employed"] = (
        df["days_last_phone_change"]
        / df["days_employed"].replace(0, np.nan)
    )

    df["pctg_days_employed"] = (
        df["days_employed"]
        / df["days_birth"].replace(0, np.nan)
    )

    # Binary features
    df["long_employment"] = (
        df["days_employed"] < -2000
    ).astype(int)

    df["retirement_age"] = (
        df["days_birth"] < -14000
    ).astype(int)

    # External source interaction
    df["ext_sources_prod"] = (
        df["ext_source_1"]
        * df["ext_source_2"]
        * df["ext_source_3"]
    )

    # Final/effective version from the original code
    df["ratio_amt_annuity_to_amt_income_total"] = (
        df["amt_annuity"]
        / (1 + df["amt_income_total"])
    )

    df["ratio_amt_credit_to_amt_goods_price"] = (
        df["amt_credit"]
        / df["amt_goods_price"].replace(0, np.nan)
    )

    return df

In [201]:
print(f"Train data shape before feature engineering: {X.shape}")
print(f"Test data shape before feature engineering: {X_test.shape}")

Train data shape before feature engineering: (292135, 121)
Test data shape before feature engineering: (15376, 121)


In [202]:
# initiate feature engineering
X = feature_engineering(X)
X_test = feature_engineering(X_test)

In [203]:
print(f"Train data shape after feature engineering: {X.shape}")
print(f"Test data shape after feature engineering: {X_test.shape}")

Train data shape after feature engineering: (292135, 175)
Test data shape after feature engineering: (15376, 175)


In [204]:
num_cols = X.select_dtypes(include=["int", "float"]).columns.to_list()
cat_cols = X.select_dtypes(exclude=["int", "float"]).columns.to_list()

print(f"Numerical Columns Count: {len(num_cols)}")
print(f"Categorical Columns Count: {len(cat_cols)}")

Numerical Columns Count: 164
Categorical Columns Count: 11


### Model Experimentation


#### Mlflow Setup


In [205]:
class MlflowConfig:
    experiment_name = "home_credit_baseline"
    version = "_v6"

In [206]:
mlflow.set_experiment(MlflowConfig.experiment_name)

<Experiment: artifact_location='file:c:/Project_Home_Credit/notebooks/model_training/mlruns/1', creation_time=1787055586649, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1787055586649, lifecycle_stage='active', name='home_credit_baseline', tags={}, trace_location=None, workspace='default'>

#### 1. HistGradientBoostingClassifier


In [207]:
RUN_NAME = "hgb" + MlflowConfig.version

In [ ]:
# Set category dtype for native handling on both splits
for col in cat_cols:
    X[col] = X[col].astype("category")
    X_test[col] = X_test[col].astype("category")

# Track fold validation scores and accumulate test predictions
test_probs = np.zeros(len(X_test), dtype=np.float64)
fold_loglosses = np.zeros(Config.n_folds, dtype=np.float64)
fold_roc_aucs = np.zeros(Config.n_folds, dtype=np.float64)

skf = StratifiedKFold(n_splits=Config.n_folds, shuffle=True, random_state=Config.seed)

with mlflow.start_run(run_name=RUN_NAME):
    mlflow.set_tags(
        {
            "model_family": "HistGradientBoostingClassifier",
            "validation": f"{Config.n_folds}_fold_stratified_cv",
        }
    )
    mlflow.log_params(PARAMS.hgb)
    mlflow.log_dict({"feature_names": X.columns.tolist()}, "feature_names.json")

    for fold, (train_indices, valid_indices) in tqdm(
        enumerate(skf.split(X, y)), desc="Model Training", total=Config.n_folds
    ):
        X_train, y_train = X.iloc[train_indices], y.iloc[train_indices]
        X_valid, y_valid = X.iloc[valid_indices], y.iloc[valid_indices]

        # Train model
        model = HistGradientBoostingClassifier(**PARAMS.hgb)
        model.fit(X_train, y_train)

        # Validation fold predictions for tracking fold stability/std
        fold_valid_probs = model.predict_proba(X_valid)[:, 1]
        fold_loglosses[fold] = log_loss(y_valid, fold_valid_probs)
        fold_roc_aucs[fold] = roc_auc_score(y_valid, fold_valid_probs)

        # Accumulate ensembled test predictions across folds
        test_probs += model.predict_proba(X_test)[:, 1] / Config.n_folds

        # Clean fold memory
        del model, X_train, y_train, X_valid, y_valid, fold_valid_probs
        gc.collect()

    # Final Test Set Metrics
    test_binary_preds = (test_probs >= 0.5).astype(np.int32)

    mlflow.log_metric("test_roc_auc", float(roc_auc_score(y_test, test_probs)))
    mlflow.log_metric("test_logloss", float(log_loss(y_test, test_probs)))
    mlflow.log_metric(
        "test_precision",
        float(precision_score(y_test, test_binary_preds, zero_division=0)),
    )
    mlflow.log_metric(
        "test_recall", float(recall_score(y_test, test_binary_preds, zero_division=0))
    )

    # Fold standard deviations
    mlflow.log_metric("std_roc_auc", float(np.std(fold_roc_aucs)))
    mlflow.log_metric("std_logloss", float(np.std(fold_loglosses)))

# Final cleanup
del test_probs, test_binary_preds, fold_loglosses, fold_roc_aucs, skf
gc.collect()

Model Training:   0%|          | 0/5 [00:00<?, ?it/s]

#### 2. LightGBM Dataset


In [ ]:
RUN_NAME = "lgbm" + MlflowConfig.version

In [ ]:
# Set category dtype for native handling on both splits
for col in cat_cols:
    X[col] = X[col].astype("category")
    X_test[col] = X_test[col].astype("category")

# Track fold validation scores and accumulate test predictions
test_probs = np.zeros(len(X_test), dtype=np.float64)
fold_loglosses = np.zeros(Config.n_folds, dtype=np.float64)
fold_roc_aucs = np.zeros(Config.n_folds, dtype=np.float64)

skf = StratifiedKFold(n_splits=Config.n_folds, shuffle=True, random_state=Config.seed)

with mlflow.start_run(run_name=RUN_NAME):
    mlflow.set_tags(
        {
            "model_family": "LightGBM_Native",
            "validation": f"{Config.n_folds}_fold_stratified_cv",
        }
    )
    mlflow.log_params(PARAMS.lgbm)
    mlflow.log_dict({"feature_names": X.columns.tolist()}, "feature_names.json")

    for fold, (train_indices, valid_indices) in tqdm(
        enumerate(skf.split(X, y)), desc="Model Training", total=Config.n_folds
    ):
        X_train, y_train = X.iloc[train_indices], y.iloc[train_indices]
        X_valid, y_valid = X.iloc[valid_indices], y.iloc[valid_indices]

        # Datasets
        dtrain = lgb.Dataset(
            X_train, label=y_train, categorical_feature=cat_cols, free_raw_data=False
        )

        dvalid = lgb.Dataset(
            X_valid,
            label=y_valid,
            reference=dtrain,
            categorical_feature=cat_cols,
            free_raw_data=False,
        )

        # Train with early stopping
        booster = lgb.train(
            params=PARAMS.lgbm,
            train_set=dtrain,
            num_boost_round=1200,
            valid_sets=[dvalid],
            callbacks=[
                lgb.early_stopping(stopping_rounds=100, verbose=False),
                lgb.log_evaluation(period=0),
            ],
        )

        # Validation fold predictions for tracking fold stability/std
        fold_valid_probs = booster.predict(
            X_valid, num_iteration=booster.best_iteration
        )
        fold_loglosses[fold] = log_loss(y_valid, fold_valid_probs)
        fold_roc_aucs[fold] = roc_auc_score(y_valid, fold_valid_probs)

        # Accumulate ensembled test predictions across folds
        test_probs += (
            booster.predict(X_test, num_iteration=booster.best_iteration)
            / Config.n_folds
        )

        # Clean fold memory
        del (
            dtrain,
            dvalid,
            booster,
            X_train,
            y_train,
            X_valid,
            y_valid,
            fold_valid_probs,
        )
        gc.collect()

    # Final Test Set Metrics
    test_binary_preds = (test_probs >= 0.5).astype(np.int32)

    mlflow.log_metric("test_roc_auc", float(roc_auc_score(y_test, test_probs)))
    mlflow.log_metric("test_logloss", float(log_loss(y_test, test_probs)))
    mlflow.log_metric(
        "test_precision",
        float(precision_score(y_test, test_binary_preds, zero_division=0)),
    )
    mlflow.log_metric(
        "test_recall", float(recall_score(y_test, test_binary_preds, zero_division=0))
    )

    # Fold standard deviations
    mlflow.log_metric("std_roc_auc", float(np.std(fold_roc_aucs)))
    mlflow.log_metric("std_logloss", float(np.std(fold_loglosses)))

# Final cleanup
del test_probs, test_binary_preds, fold_loglosses, fold_roc_aucs, skf
gc.collect()

Model Training:   0%|          | 0/5 [00:00<?, ?it/s]

18

#### 3. XGBoost DMatrix


In [ ]:
RUN_NAME = "xgb" + MlflowConfig.version

In [ ]:
# Set category dtype for native handling on both splits
for col in cat_cols:
    X[col] = X[col].astype("category")
    X_test[col] = X_test[col].astype("category")

# Track fold validation scores and accumulate test predictions
test_probs = np.zeros(len(X_test), dtype=np.float64)
fold_loglosses = np.zeros(Config.n_folds, dtype=np.float64)
fold_roc_aucs = np.zeros(Config.n_folds, dtype=np.float64)

skf = StratifiedKFold(n_splits=Config.n_folds, shuffle=True, random_state=Config.seed)

with mlflow.start_run(run_name=RUN_NAME):
    mlflow.set_tags(
        {
            "model_family": "XGBoost_Native",
            "validation": f"{Config.n_folds}_fold_stratified_cv",
        }
    )
    mlflow.log_params(PARAMS.xgb)
    mlflow.log_dict({"feature_names": X.columns.tolist()}, "feature_names.json")

    for fold, (train_indices, valid_indices) in tqdm(
        enumerate(skf.split(X, y)), desc="Model Training", total=Config.n_folds
    ):
        X_train = X.iloc[train_indices].copy()
        y_train = y.iloc[train_indices].copy()
        X_valid = X.iloc[valid_indices].copy()
        y_valid = y.iloc[valid_indices].copy()

        # Target Encoding
        te_cv = KFold(n_splits=3, shuffle=True, random_state=Config.seed)
        encoder = TargetEncoder(
            target_type="binary",
            cv=te_cv,
        )

        X_train[ohe_cols] = encoder.fit_transform(X_train[ohe_cols], y_train)
        X_valid[ohe_cols] = encoder.transform(X_valid[ohe_cols])
        
        # Create test copy
        X_test_fold = X_test.copy()
        X_test_fold[ohe_cols] = encoder.transform(X_test_fold[ohe_cols])

        # Cast to float
        X_train[ohe_cols] = X_train[ohe_cols].astype(float)
        X_valid[ohe_cols] = X_valid[ohe_cols].astype(float)
        X_test_fold[ohe_cols] = X_test_fold[ohe_cols].astype(float)

        sample_weights = compute_sample_weight(
            class_weight="balanced",
            y=y_train,
        )

        # Native DMatrix initialization
        dtrain = xgb.DMatrix(
            X_train, 
            label=y_train, 
            enable_categorical=True,
            weight=sample_weights
        )

        dvalid = xgb.DMatrix(
            X_valid, 
            label=y_valid, 
            enable_categorical=True
        )
        
        # Native DMatrix for the test fold
        dtest_fold = xgb.DMatrix(
            X_test_fold, 
            enable_categorical=True
        )

        # Train with early stopping
        booster = xgb.train(
            params=PARAMS.xgb,
            dtrain=dtrain,
            num_boost_round=1200,
            evals=[(dvalid, "valid")],
            early_stopping_rounds=100,
            verbose_eval=False,
        )

        # Validation fold predictions for tracking fold stability/std
        fold_valid_probs = booster.predict(dvalid)
        fold_loglosses[fold] = log_loss(y_valid, fold_valid_probs)
        fold_roc_aucs[fold] = roc_auc_score(y_valid, fold_valid_probs)

        # Accumulate ensembled test predictions across folds (Using dtest_fold!)
        test_probs += booster.predict(dtest_fold) / Config.n_folds

        # Clean fold memory
        del (
            dtrain,
            dvalid,
            dtest_fold,
            booster,
            X_train,
            y_train,
            X_valid,
            y_valid,
            X_test_fold,
            fold_valid_probs,
            sample_weights,
            encoder
        )
        gc.collect()

    # Final Test Set Metrics
    test_binary_preds = (test_probs >= 0.5).astype(np.int32)

    mlflow.log_metric("test_roc_auc", float(roc_auc_score(y_test, test_probs)))
    mlflow.log_metric("test_logloss", float(log_loss(y_test, test_probs)))
    mlflow.log_metric(
        "test_precision",
        float(precision_score(y_test, test_binary_preds, zero_division=0)),
    )
    mlflow.log_metric(
        "test_recall", float(recall_score(y_test, test_binary_preds, zero_division=0))
    )

    # Fold standard deviations
    mlflow.log_metric("std_roc_auc", float(np.std(fold_roc_aucs)))
    mlflow.log_metric("std_logloss", float(np.std(fold_loglosses)))

# Final cleanup
del test_probs, test_binary_preds, fold_loglosses, fold_roc_aucs, skf
gc.collect()

Model Training:   0%|          | 0/5 [00:00<?, ?it/s]

18

#### 4. CatBoost Pool


In [ ]:
RUN_NAME = "cb" + MlflowConfig.version

In [ ]:
# Handle NaNs and cast to str for CatBoost on both splits
for col in cat_cols:
    X[col] = X[col].astype(object).fillna("missing").astype(str)
    X_test[col] = X_test[col].astype(object).fillna("missing").astype(str)

# Native Pool for test set evaluation
test_pool = Pool(X_test, cat_features=cat_cols)

# Track fold validation scores and accumulate test predictions
test_probs = np.zeros(len(X_test), dtype=np.float64)
fold_loglosses = np.zeros(Config.n_folds, dtype=np.float64)
fold_roc_aucs = np.zeros(Config.n_folds, dtype=np.float64)

skf = StratifiedKFold(
    n_splits=Config.n_folds, 
    shuffle=True, 
    random_state=Config.seed,
    )

with mlflow.start_run(run_name=RUN_NAME):
    mlflow.set_tags(
        {
            "model_family": "CatBoost_Native_Pool",
            "validation": f"{Config.n_folds}_fold_stratified_cv",
        }
    )
    mlflow.log_params(PARAMS.cb)
    mlflow.log_dict({"feature_names": X.columns.tolist()}, "feature_names.json")

    for fold, (train_indices, valid_indices) in tqdm(
        enumerate(skf.split(X, y)), desc="Model Training", total=Config.n_folds
    ):
        X_train, y_train = X.iloc[train_indices], y.iloc[train_indices]
        X_valid, y_valid = X.iloc[valid_indices], y.iloc[valid_indices]

        # Native pools
        train_pool = Pool(X_train, label=y_train, cat_features=cat_cols)
        valid_pool = Pool(X_valid, label=y_valid, cat_features=cat_cols)

        # Initialize and fit
        model = CatBoostClassifier(**PARAMS.cb)
        model.fit(train_pool, eval_set=valid_pool, use_best_model=True, verbose=False)

        # Validation fold predictions for tracking fold stability/std
        fold_valid_probs = model.predict_proba(valid_pool)[:, 1]
        fold_loglosses[fold] = log_loss(y_valid, fold_valid_probs)
        fold_roc_aucs[fold] = roc_auc_score(y_valid, fold_valid_probs)

        # Accumulate ensembled test predictions across folds
        test_probs += model.predict_proba(test_pool)[:, 1] / Config.n_folds

        # Clean fold memory
        del (
            train_pool,
            valid_pool,
            model,
            X_train,
            y_train,
            X_valid,
            y_valid,
            fold_valid_probs,
        )
        gc.collect()

    # Final Test Set Metrics
    test_binary_preds = (test_probs >= 0.5).astype(np.int32)

    mlflow.log_metric("test_roc_auc", float(roc_auc_score(y_test, test_probs)))
    mlflow.log_metric("test_logloss", float(log_loss(y_test, test_probs)))
    mlflow.log_metric(
        "test_precision",
        float(precision_score(y_test, test_binary_preds, zero_division=0)),
    )
    mlflow.log_metric(
        "test_recall", float(recall_score(y_test, test_binary_preds, zero_division=0))
    )

    # Fold standard deviations
    mlflow.log_metric("std_roc_auc", float(np.std(fold_roc_aucs)))
    mlflow.log_metric("std_logloss", float(np.std(fold_loglosses)))

# Final cleanup
del test_probs, test_binary_preds, fold_loglosses, fold_roc_aucs, skf, test_pool
gc.collect()

Model Training:   0%|          | 0/5 [00:00<?, ?it/s]

Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


18